<a href="https://colab.research.google.com/github/Donthula-Vyshnavi/ACCENTUREDT/blob/main/PySparkDay1%262%263_SF_GCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install pyspark pyarrow --quiet

In [26]:
from pyspark.sql import SparkSession


In [27]:
spark = SparkSession.builder.appName("RDD_Basics").getOrCreate()
sc = spark.sparkContext

In [28]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])


In [29]:
print(rdd1.collect())

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [30]:
rdd2 = sc.textFile("/yellow_tripdata_2019-05.csv")

In [34]:
print(rdd2.first())
print(rdd2.take(5))

ValueError: RDD is empty

In [35]:
print(rdd2.count())

0


# Transformations:
- Create a new RDD from existing RDD
example :
- maps: changes each element ,
- filter: removes the unwanted data,
- flatmap: breaks up list of words into individual words

- Lazy evaluation: transformations would not actually run until an action is called

# Action
- Trigger all the transformations.
examples:
- count() : num of ele
- collect(): give the data in form of list
- take(n) : gives first n ele
- reduce - combines the data into one


In [36]:
rdd3 = sc.parallelize(["apple", "banana", "cherry"])
print(rdd3.map(lambda x: x.upper()).collect())

['APPLE', 'BANANA', 'CHERRY']


# **map and flatmap difference** -- is map it flattens the results

In [37]:
rdd4 =sc.parallelize(["hello world", "how are you"])
print(rdd4.map(lambda x: x.split(" ")).collect())

[['hello', 'world'], ['how', 'are', 'you']]


In [38]:
rdd4 =sc.parallelize(["hello world", "how are you"])
words = rdd4.flatMap(lambda x: x.split(" "))
print(words.collect())

['hello', 'world', 'how', 'are', 'you']


In [ ]:
## condition

In [39]:
rdd55 = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(rdd55.filter(lambda x: x % 2 == 0).collect())

[2, 4, 6, 8, 10]


In [40]:
filter_rdd3 = rdd3.filter(lambda x: "a" in x)
print(filter_rdd3.collect())

['apple', 'banana']


# Actions: collect, count, first, saveAsTextFile

# Transformations : map(), flatmap(), filter(), reduceBykey(), groupByKey(), join(), union(), distinct()

In [41]:
# Pair RDD, reducebykey, groupbykey, sortbykey
rdd = sc.parallelize([("a",1),("b",2),("a",3)])
grouped_rdd1 = rdd.groupByKey()
x = grouped_rdd1.collect()
print(x)

[('b', <pyspark.resultiterable.ResultIterable object at 0x7e483bbccdd0>), ('a', <pyspark.resultiterable.ResultIterable object at 0x7e483bbcd700>)]


In [44]:
for i in x:
  print(i[0], [j for j in i[1]])

b [2]
a [1, 3]


In [43]:
rdd = sc.parallelize([("a",1),("b",2),("a",3)])
reduced_rdd1 = rdd.reduceByKey(lambda x,y: x+y)
print(reduced_rdd1.collect())

[('b', 2), ('a', 4)]


In [45]:
rdd = sc.parallelize([("a",1),("b",2),("a",3)])
sort_rdd1 = rdd.sortByKey(lambda x:x[1])
print(sort_rdd1.collect())

[('a', 1), ('a', 3), ('b', 2)]


In [47]:
sample_data = [("cash", 10.50), ("card", 15.50),("cash", 20.50), ("card", 15.50),("cash", 30.50), ("card", 15.50), ("Digital", 12.80)]
rdd = sc.parallelize(sample_data)
group_rdd = rdd.groupByKey()
print(group_rdd.collect())

[('cash', <pyspark.resultiterable.ResultIterable object at 0x7e483bd756d0>), ('card', <pyspark.resultiterable.ResultIterable object at 0x7e483bd749e0>), ('Digital', <pyspark.resultiterable.ResultIterable object at 0x7e483bd75f10>)]


In [48]:
for i in group_rdd.collect():
  print(i[0], [j for j in i[1]])

cash [10.5, 20.5, 30.5]
card [15.5, 15.5, 15.5]
Digital [12.8]


In [49]:
# cache() and persist()

from time import time

rdd = sc.parallelize(range(1,5000000))\
        .map(lambda x: x*2)
start = time()
print(rdd.count())
print("time for count", time() - start)
print(rdd.sum())
print("time for sum", time() - start)

4999999
time for count 2.9048666954040527
24999995000000
time for sum 5.013505697250366


In [50]:
rdd_cached = rdd.cache()
start = time()
print(rdd_cached.count())
print("time for count", time() - start)
print(rdd_cached.sum())
print("time for sum", time() - start)

4999999
time for count 3.1616313457489014
24999995000000
time for sum 3.819291114807129


In [51]:
from pyspark.storagelevel import StorageLevel
rdd_persisted = sc.parallelize(range(1,500000)).map(lambda x : x *x).persist(StorageLevel.MEMORY_AND_DISK)
start = time()
print(rdd_persisted.count())
print("time for count", time() - start)
print(rdd_persisted.sum())
print("time for sum", time() - start)

499999
time for count 0.7274255752563477
41666541666750000
time for sum 1.0599219799041748


In [52]:
#accululators and broadcast variables
accum= sc.accumulator(0)
rdd = sc.parallelize([1,2,3,4,5])
rdd.foreach(lambda x: accum.add(x))
print(accum.value)

15


In [53]:
broadcast_data = sc.broadcast([1,2,3,4,5])
print(broadcast_data.value)

[1, 2, 3, 4, 5]


# **Dataframes**

In [54]:
from pyspark.sql import SparkSession

In [55]:
spark = SparkSession.builder.appName("DataframesDemo").getOrCreate()

In [56]:
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)
df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [57]:
df1 = spark.read.csv("/yellow_tripdata_2019-05.csv", header=True, inferSchema=True)
df1.show()

++
||
++
++



In [58]:
rdd = spark.sparkContext.parallelize(data)
df2 = rdd.toDF(["name", "rollno"]
               )
df2.show()

+-------+------+
|   name|rollno|
+-------+------+
|  Alice|    25|
|    Bob|    30|
|Charlie|    35|
+-------+------+



In [59]:
#transformations : select, filter, withColumn, drop, groupby, orderby

In [60]:
df.select("name").show()

+-------+
|   name|
+-------+
|  Alice|
|    Bob|
|Charlie|
+-------+



In [61]:
df.filter(df["age"] > 30).show()

+-------+---+
|   Name|Age|
+-------+---+
|Charlie| 35|
+-------+---+



In [62]:
df.withColumn("age2", df["age"] + 1).show()

+-------+---+----+
|   Name|Age|age2|
+-------+---+----+
|  Alice| 25|  26|
|    Bob| 30|  31|
|Charlie| 35|  36|
+-------+---+----+



In [63]:
df.drop("age2").show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [65]:
df.orderBy("age").show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



**Actions - show, collect, count, sum, take, write**

In [64]:
df.count()

3

In [66]:
df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



# YELLOW TRIP DATA – DF PRACTICE QUESTIONS
Filter out the header.


Take the first 5 records.


Count total number of records.


Filter records that contain “N”.


Count how many records contain “Card”.


Map each row to its length.


Reduce to find:


Total length of all records


Maximum record length


Count records longer than 250 characters.


In [67]:
#1.Filter out the header.
df = spark.read.csv("/yellow_tripdata_2019-05.csv", header=True, inferSchema=True)
df.show()

++
||
++
++



In [69]:
#2.Take the first 5 records.
df.take(5)

[]

In [71]:
#3.Count total number of records.
df.count()

0

In [73]:
#4.Filter records that contain “N”.
df.filter(df["store_and_fwd_flag"].contains("N")).show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `store_and_fwd_flag` cannot be resolved. Did you mean one of the following? []. SQLSTATE: 42703

In [74]:
#5.Count how many records contain “Card”.
df.filter(df["payment_type"].contains("Card")).count()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `payment_type` cannot be resolved. Did you mean one of the following? []. SQLSTATE: 42703

In [75]:
#6.Map each row to its length.
df.rdd.map(lambda x: len(x)).collect()

[]

In [76]:
#7.Reduce to find:Total length of all records
df.rdd.map(lambda x: len(x)).sum()



0

In [77]:
#8.Reduce to find:Maximum record length
df.rdd.map(lambda x: len(x)).max()


ValueError: Can not reduce() empty RDD

In [78]:
#9.Reduce to find:Count records longer than 250 characters.
df.rdd.filter(lambda x: len(x) > 250).count()

0

# PRACTICE QUESTIONS


In [79]:
#1 A take first 10 rows
df.take(10)


[]

In [80]:
df.printSchema()

root



In [81]:
#1 B show me the schema and data types of the df DataFrame.
df.printSchema()


root



In [82]:
#1C count number of rows and columns
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 0
Number of columns: 0


In [84]:
#2 Create a new DataFrame with only these columns:VendorID, tpep_pickup_datetime, trip_distance, fare_amount, total_amount, payment_type
new_df = df.select("VendorID", "tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount", "payment_type")
new_df.show(10)




{"ts": "2026-07-31 16:20:15.550", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `VendorID` cannot be resolved.  SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor70.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o727.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `VendorID` cannot be resolved.  SQLSTATE: 42703;\n'Project ['VendorID, 'tpep_pickup_datetime, 'trip_distance, 'fare_amount, 'total_amount, 'payment_type]\n+- Relation [] csv\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:401)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.

AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `VendorID` cannot be resolved.  SQLSTATE: 42703;
'Project ['VendorID, 'tpep_pickup_datetime, 'trip_distance, 'fare_amount, 'total_amount, 'payment_type]
+- Relation [] csv


In [ ]:
#3 Filter all trips where trip_distance is greater than 10 miles.How many such trips exist?
filtered_df = df.filter(df["trip_distance"] > 10)
filtered_df.count()

In [85]:
#4 Find all trips where:passenger_count is greater than 2 AND payment_type is 1 (Credit Card) AND fare_amount is greater than 20 Show the result and count total matching rows.
filtered_df = df.filter((df["passenger_count"] > 2) & (df["payment_type"] == 1) & (df["fare_amount"] > 20))
filtered_df.show()
filtered_df.count()


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `passenger_count` cannot be resolved. Did you mean one of the following? []. SQLSTATE: 42703

In [86]:
#5 Add a new column called tip_percentage that shows the tip as a percentage of fare_amount:tip_percentage = (tip_amount / fare_amount) * 100  .Round it to 2 decimal places. Show top 10 rows.
from pyspark.sql.functions import round, when, col
df = df.withColumn("tip_percentage",
                    when(df["fare_amount"] == 0, 0.0)  # Handle division by zero
                    .otherwise(round((df["tip_amount"] / df["fare_amount"]) * 100, 2)))
df.show(10)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `fare_amount` cannot be resolved. Did you mean one of the following? []. SQLSTATE: 42703

In [ ]:
#6.Rename and Drop Columns Rename tpep_pickup_datetime to pickup_time Rename tpep_dropoff_datetime to dropoff_time Drop the ehail_fee column (it is all nulls anyway)Show the resulting schema.Topics: withColumnRenamed(), drop()
df = df.withColumnRenamed("tpep_pickup_datetime", "pickup_time")
df = df.withColumnRenamed("tpep_dropoff_datetime", "dropoff_time")
df = df.drop("ehail_fee")
df

In [ ]:
#Q7. Sort the Data:Sort the DataFrame by total_amount in descending order.Show the top 5 most expensive trips with columns:VendorID, trip_distance, fare_amount, tip_amount, total_amount.Topics: orderBy(), desc()
from pyspark.sql.functions import desc
df.orderBy(desc("total_amount")).select("VendorID", "trip_distance", "fare_amount", "tip_amount", "total_amount").show(5)

In [32]:
#Q8. Distinct Values:Find all distinct values of payment_type and RatecodeID.Also find the total number of distinct PULocationID pickup locations.Topics: distinct(), select().distinct(), dropDuplicates()
df.select("payment_type").distinct().show()
df.select("RatecodeID").distinct().show()
df.select("PULocationID").distinct().count()


{"ts": "2026-07-31 16:17:23.551", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `payment_type` cannot be resolved.  SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o285.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `payment_type` cannot be resolved.  SQLSTATE: 42703;\n'Project ['payment_type]\n+- Relation [] csv\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:401)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnre

AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `payment_type` cannot be resolved.  SQLSTATE: 42703;
'Project ['payment_type]
+- Relation [] csv


In [ ]:
#Q9a. Basic Aggregations:Calculate the following for the entire dataset:Total number of trips.
from pyspark.sql.functions import col, sum, avg, max, min, count
df.select(count("*").alias("Total number of trips")).show()

In [ ]:
#9b.Average trip_distance
df.select(avg("trip_distance").alias("Average trip_distance")).show()

In [ ]:
#9c.Maximum fare_amount
df.select(max("fare_amount").alias("Maximum fare_amount")).show()

In [ ]:
#9d.Minimum fare_amount
df.select(min("fare_amount").alias("Minimum fare_amount")).show()

In [ ]:
#9e.Total revenue (sum of total_amount)
df.select(sum("total_amount").alias("Total revenue")).show()

In [ ]:
#Q10a. GroupBy — Vendor Analysis .Total number of trips per vendor
df.groupBy("VendorID").agg(count("*").alias("Total number of trips")).show()


In [ ]:
#Q10B. Average fare per vendor
df.groupBy("VendorID").agg(avg("fare_amount").alias("Average fare")).show()

In [ ]:
#Q10C.Average trip distance per vendor
df.groupBy("VendorID").agg(avg("trip_distance").alias("Average trip distance")).show()

In [ ]:
#Q10D.Total revenue per vendor
df.groupBy("VendorID").agg(sum("total_amount").alias("Total revenue")).show()

In [ ]:
#Q11. GroupBy — Payment Type Analysis
#Group by payment_type and find:
#Number of trips per payment type
#Average tip amount per payment type
#Total amount collected per payment type
#Which payment type generates the most tips?
#Topics: groupBy(), agg(), orderBy()
from pyspark.sql.functions import col, sum, avg, max, min, count
df.groupBy("payment_type").agg(
    count("*").alias("Number of trips"),
    avg("tip_amount").alias("Average tip amount"),
    sum("total_amount").alias("Total amount collected")
).orderBy(col("Total amount collected").desc()).show()


In [ ]:
#Q12.The ehail_fee column is completely null.Topics: isNull(), fillna(), dropna(), filter()
df.filter(df["ehail_fee"].isNull()).show()



In [ ]:
#12.Drop the ehail_fee column
df = df.drop("ehail_fee")
df.show()

In [ ]:
#12.Fill any remaining nulls in congestion_surcharge with 0
df = df.fillna({"congestion_surcharge": 0})
df.show()

In [ ]:
#12.Drop any rows where passenger_count is null or 0
df = df.dropna(subset=["passenger_count"])
df.show()

In [ ]:
#13.DATETIMEOPERATIONS
#From lpep_pickup_datetime:
#Extract the hour of pickup into a new column pickup_hour
#Extract the day of week into a new column day_of_week (1=Sunday)
#Extract the date only into a new column pickup_date
#Then find which hour of the day has the most trips.
from pyspark.sql.functions import hour, dayofweek, to_date
df = df.withColumn("pickup_hour", hour(df["pickup_time"]))
df = df.withColumn("day_of_week", dayofweek(df["pickup_time"]))
df = df.withColumn("pickup_date", to_date(df["pickup_time"]))
df.groupBy("pickup_hour").count().orderBy("count", ascending=False).show(1)


In [ ]:
#13.Extract the date only into a new column pickup_date
from pyspark.sql.functions import to_date
df = df.withColumn("pickup_date", to_date(df["pickup_time"]))
df.show()

In [ ]:
#13.#Extract the day of week into a new column day_of_week (1=Sunday)
from pyspark.sql.functions import dayofweek
df = df.withColumn("day_of_week", dayofweek(df["pickup_time"]))
df.show()

In [ ]:
#13.Extract the date only into a new column pickup_date
from pyspark.sql.functions import to_date
df = df.withColumn("pickup_date", to_date(df["pickup_time"]))
df.show()

In [ ]:
#13.#Then find which hour of the day has the most trips.
from pyspark.sql.functions import hour
df.groupBy(hour("pickup_time")).count().orderBy("count", ascending=False).show(1)


In [33]:
#14 Calculate the trip duration in minutes as a new column trip_duration_mins:trip_duration = dropoff_time - pickup_time (in minutes)
#Then find:Average trip duration
#Top 5 longest trips by duration
#Topics: unix_timestamp(), withColumn(), arithmetic on timestamps
from pyspark.sql.functions import unix_timestamp, col
df = df.withColumn("trip_duration_mins", (unix_timestamp(col("dropoff_time")) - unix_timestamp(col("pickup_time"))) / 60)
df.show()

{"ts": "2026-07-31 16:17:23.819", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `dropoff_time` cannot be resolved.  SQLSTATE: 42703", "context": {"file": "line 6 in cell [33]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o285.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `dropoff_time` cannot be resolved.  SQLSTATE: 42703;\n'Project ['`/`('`-`('unix_timestamp('dropoff_time, yyyy-MM-dd HH:mm:ss), 'unix_timestamp('pickup_time, yyyy-MM-dd HH:mm:ss)), 60) AS trip_duration_mins#509]\n+- Relation [] csv\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:401)\n\tat org.apache.spark.sql.catalyst.analysis

AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `dropoff_time` cannot be resolved.  SQLSTATE: 42703;
'Project ['`/`('`-`('unix_timestamp('dropoff_time, yyyy-MM-dd HH:mm:ss), 'unix_timestamp('pickup_time, yyyy-MM-dd HH:mm:ss)), 60) AS trip_duration_mins#509]
+- Relation [] csv


In [ ]:
##14 Then find:Average trip duration
from pyspark.sql.functions import avg
df.select(avg("trip_duration_mins").alias("Average trip duration")).show()


In [ ]:
#14.Top 5 longest trips by duration
from pyspark.sql.functions import desc
df.orderBy(desc("trip_duration_mins")).select("trip_duration_mins").show(5)


In [ ]:
#Q15. String Operations:
#Convert store_and_fwd_flag values to full words:Y → "Yes" and N → "No" using when and otherwise
#Topics: when(), otherwise(), col()
from pyspark.sql.functions import when, col
df = df.withColumn("store_and_fwd_flag", when(col("store_and_fwd_flag") == "Y", "Yes").otherwise("No"))
df.show()



In [ ]:
#16 Then count how many trips fall in each category
df.groupBy("store_and_fwd_flag").count().show()

In [ ]:
#Q17. Pivot Table
#Create a pivot table showing:
#Rows: VendorID
#Columns: payment_type (1, 2, 3, 4)
#Values: total number of trips
#Topics: groupBy().pivot().count()
from pyspark.sql.functions import count
df.groupBy("VendorID").pivot("payment_type").agg(count(col("VendorID")).alias("total_trips")).show()




In [ ]:
#Q18. Window Functions — Running Total
#Using Window functions:
#Rank trips by total_amount within each VendorID (highest first)
#Topics: Window, rank(), partitionBy(), orderBy()
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
windowSpec = Window.partitionBy("VendorID").orderBy(desc("total_amount"))
df = df.withColumn("rank", row_number().over(windowSpec))
df.show()






In [ ]:
#Add a column rank showing this rank
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
windowSpec = Window.partitionBy("VendorID").orderBy(desc("total_amount"))
df = df.withColumn("rank", row_number().over(windowSpec))
df.show()
#Show top 3 trips per vendor
from pyspark.sql.functions import desc
df.filter(df["rank"] <= 3).show()


LEVEL 3 — Advanced (Q19 to Q27)
Joins, UDFs, Performance, SQL, Schema

In [ ]:
#Q19. Create a Lookup Table & Join
#Create a small DataFrame manually:
#payment_type | payment_name
#1            | Credit Card
#2            | Cash
#3            | No Charge
#4            | Dispute
#5            | Unknown
#Join it with the taxi DataFrame on payment_type.
#Show trip_distance, total_amount, payment_name for the first 10 rows.
#Topics: join(), creating DataFrames manually, inner join
lookup_data = [
    (1, "Credit Card"),
    (2, "Cash"),
    (3, "No Charge"),
    (4, "Dispute"),
    (5, "Unknown")
]
lookup_df = spark.createDataFrame(lookup_data, ["payment_type", "payment_name"])
df = df.join(lookup_df, "payment_type")
df.select("trip_distance", "total_amount", "payment_name").show(10)


In [ ]:
#drop RatecodeName
df = df.drop("RatecodeName")
df.show()

In [ ]:
#Q20. Left Join — Find Unmatched Records
#Create a second lookup table for RatecodeID but only include codes 1, 2, 3 (leave out 4, 5, 6).
#Do a left join with the taxi data.
#Find all trips whose RatecodeID did NOT match (null after join).
#Topics: left join, isNull() after join, unmatched records.
lookup_data = [
    (1, "Standard rate"),
    (2, "JFK"),
    (3, "Newark")
]
lookup_df = spark.createDataFrame(lookup_data, ["RatecodeID", "RatecodeName"])
df = df.join(lookup_df, "RatecodeID", "left_outer")
df.filter(df["RatecodeName"].isNull()).show()





In [ ]:
#Q21. UDF — Custom Fare Category
#Write a Python UDF that categorizes total_amount into:Less than 10 → "Low",10 to 30 → "Medium",More than 30 → "High"
#Apply it to the DataFrame as a new column fare_category.
#Count trips in each category.
#Topics: udf(), StringType(), withColumn()
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
#Write a Python UDF that categorizes total_amount into:Less than 10 → "Low",10 to 30 → "Medium",More than 30 → "High"
def categorize_fare(total_amount):
    if total_amount < 10:
        return "Low"
    elif 10 <= total_amount <= 30:
        return "Medium"
    else:
        return "High"
#Apply it to the DataFrame as a new column fare_category.
categorize_udf = udf(categorize_fare, StringType())
df = df.withColumn("fare_category", categorize_udf(df["total_amount"]))
df.show()
##Count trips in each category.
df.groupBy("fare_category").count().show()
#Topics: udf(), StringType(), withColumn()




In [ ]:
 #22.Write SQL queries to:
df.createOrReplaceTempView("yellow_tripdata_2019_05")
#Find average fare per hour of day
result = spark.sql("""
    SELECT
        HOUR(pickup_time) AS hour_of_day,
        AVG(fare_amount) AS avg_fare
    FROM yellow_tripdata_2019_05
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""")
#Find top 10 busiest pickup locations (PULocationID)
result = spark.sql("""
    SELECT
        PULocationID,
        COUNT(*) AS trip_count
    FROM yellow_tripdata_2019_05
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")
#Find total revenue per day
result = spark.sql("""
    SELECT
        pickup_date,
        SUM(total_amount) AS total_revenue
    FROM yellow_tripdata_2019_05
    GROUP BY pickup_date
    ORDER BY pickup_date
""")
result.show()

In [ ]:
import os
#Q23. Read & Write — Parquet
#Save the cleaned DataFrame as a Parquet file partitioned by VendorID
#Read it back
#Confirm the row count matches the original
#Compare file size of CSV vs Parquet
#Topics: write.parquet(), partitionBy(), read.parquet(), mode("overwrite")
df.write.parquet("yellow_tripdata_2019_05_parquet", partitionBy="VendorID", mode="overwrite")
df_parquet = spark.read.parquet("yellow_tripdata_2019_05_parquet")
print("Row count in original DataFrame:", df.count())
print("Row count in Parquet DataFrame:", df_parquet.count())

csv_size = os.path.getsize("/yellow_tripdata_2019-05.csv") # Corrected path
parquet_size = os.path.getsize("yellow_tripdata_2019_05_parquet")
print("CSV file size:", csv_size, "bytes")
print("Parquet file size:", parquet_size, "bytes")

In [ ]:
#Q24. Schema Definition — Explicit Types
#Re-load the CSV but this time define the full schema manually using StructType and StructField — with correct types:
#Timestamps for datetime columns
#DoubleType for fare/amount columns
#IntegerType for ID columns
#Compare with inferSchema=True — which is faster and why?
#Topics: StructType, StructField, TimestampType, DoubleType, IntegerType
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, IntegerType
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True)
])


In [ ]:
#Q25. Cache & Performance Test
#Run this experiment:
#WITHOUT cache — run count(), avg(trip_distance), max(total_amount) and measure total time
#WITH cache() — run the same 3 actions and measure total time
#Print the difference
#Explain WHY the second run is faster.
#Topics: cache(), persist(), time(), lazy evaluation
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from time import time

spark = SparkSession.builder.appName("DataframesDemo").getOrCreate()

# Re-initialize df for the performance test to ensure it's defined
# The original file was not found, so creating a dummy DataFrame to allow execution to proceed.
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
schema = StructType([
    StructField("trip_distance", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])
dummy_data = [(10.5, 25.0), (1.2, 5.0), (15.0, 40.0), (3.0, 10.0)]
df = spark.createDataFrame(dummy_data, schema)

# WITHOUT cache
start_time = time()
df.count()
df.select(col("trip_distance")).agg({"trip_distance": "avg"}).collect()
df.select(col("total_amount")).agg({"total_amount": "max"}).collect()
end_time = time()
time_without_cache = end_time - start_time
print("Time without cache:", time_without_cache)

# WITH cache() — run the same 3 actions and measure total time
start_time = time()
df.cache().count()
df.select(col("trip_distance")).agg({"trip_distance": "avg"}).collect()
df.select(col("total_amount")).agg({"total_amount": "max"}).collect()
end_time = time()
time_with_cache = end_time - start_time
print("Time with cache:", time_with_cache)

#Print the difference
print("Difference:", time_without_cache - time_with_cache)

In [ ]:
#Q26. Window Function — Cumulative Revenue
#Using Window functions, calculate:
#Cumulative sum of total_amount ordered by lpep_pickup_datetime
#Add it as a column cumulative_revenue
#Also add a row_number column per VendorID
#Topics: Window.orderBy(), sum().over(), row_number()
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, row_number
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, TimestampType
from datetime import datetime

# Redefining df with necessary columns for this question, as the previous df was simplified.
# This assumes 'spark' is already defined from a previous cell.
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])
dummy_data = [
    (1, datetime(2019, 5, 1, 0, 10, 0), 10.5, 25.0),
    (1, datetime(2019, 5, 1, 0, 15, 0), 1.2, 5.0),
    (2, datetime(2019, 5, 1, 0, 5, 0), 15.0, 40.0),
    (2, datetime(2019, 5, 1, 0, 20, 0), 3.0, 10.0),
    (1, datetime(2019, 5, 1, 0, 25, 0), 5.0, 12.0),
    (2, datetime(2019, 5, 1, 0, 30, 0), 7.0, 18.0)
]
df = spark.createDataFrame(dummy_data, schema)


windowSpec = Window.partitionBy("VendorID").orderBy("tpep_pickup_datetime")
df = df.withColumn("cumulative_revenue", sum("total_amount").over(windowSpec))
df = df.withColumn("row_number", row_number().over(windowSpec))
df.show()

In [87]:
#Q27. Full Pipeline — End to End
#Build a complete pipeline that:Loads the CSV with proper schema

#Drops the ehail_fee column
#Fills nulls with 0
#Adds trip_duration_mins, tip_percentage, distance_category columns
#Filters out trips with trip_distance = 0
#Groups by VendorID and distance_category — finds avg fare and total trips
#Saves the result as Parquet
#Topics: All topics combined — the full workflow





In [88]:
#Q28. Explain This Code
#Look at this code and explain what it does, why it is efficient or not, and how to improve it:
#python
#df.filter(col("trip_distance") > 5) \.filter(col("fare_amount") > 10) \.filter(col("payment_type") == 1) \.count()
#Tests: Understanding of Catalyst optimizer, predicate pushdown







In [ ]:
#Q29. Transformation or Action?
#Classify each of these as a Transformation or Action and explain why:
#df.select()         df.count()
#df.filter()         df.show()
#df.groupBy()        df.collect()
#df.withColumn()     df.write.save()
#df.orderBy()        df.cache()
#Tests: Lazy evaluation, DAG understanding



In [ ]:
#Q30. Debug This Error
#A student writes this code and gets an error. Find the bug and fix it:
#python
#df.groupBy("VendorID") \.avg("fare_amount", "trip_distance") \.withColumn("fare_per_mile",              col("avg(fare_amount)") / col("avg(trip_distance)")) \.show()
#Tests: Column naming after aggregation, alias usage



# **Spark Structured Streaming**

In [89]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StructuredStreamingWordCount") \
    .getOrCreate()


In [90]:
lines = (
    spark.readStream
         .format("text")
         .load("/content/StreamingInput/")
)


AnalysisException: [PATH_NOT_FOUND] Path does not exist: /content/StreamingInput/. SQLSTATE: 42K03

In [91]:
from pyspark.sql.functions import explode, split

words = lines.select(
    explode(split(lines.value, " ")).alias("word")
)


NameError: name 'lines' is not defined

In [ ]:
wordCounts = words.groupBy("word").count()


In [ ]:
query = (
    wordCounts.writeStream
        .outputMode("complete")
        .format("console")
        .trigger(once=True)
        .start()
)
query.awaitTermination()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("ColabStreamingWordCount") \
    .getOrCreate()

lines = spark.readStream \
    .format("text") \
    .load("/content/StreamingInput")

words = lines.select(
    explode(split(lines.value, " ")).alias("word")
)

wordCounts = words.groupBy("word").count()

query = (
    wordCounts.writeStream
        .outputMode("update")
        .format("memory")          # 🔑 KEY CHANGE
        .queryName("word_counts")  # table name
        .trigger(once=True)
        .start()
)

query.awaitTermination()


In [ ]:
spark.sql("SELECT * FROM word_counts").show()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split

# Create Spark Session
spark = SparkSession.builder \
    .appName("StructuredStreamingLogLevelCount") \
    .getOrCreate()

# Read streaming data from directory (must be a directory)
logs = (
    spark.readStream
         .format("text")
         .load("/content/StreamingInput/streaming")
)

# Extract log level (first word in each line)
logLevels = logs.select(
    split(logs.value, " ").getItem(0).alias("level")
)

# Aggregate: count logs by level
levelCounts = logLevels.groupBy("level").count()

# Write stream to MEMORY sink
query = (
    levelCounts.writeStream
        .outputMode("update")
        .format("memory")              # 🔑 CHANGE
        .queryName("log_level_counts") # table name
        .trigger(once=True)            # 🔑 CHANGE
        .start()
)

# Wait for streaming to finish
query.awaitTermination()



In [ ]:
spark.sql("SELECT * FROM log_level_counts").show()
